In [ ]:
%load_ext autoreload
%autoreload 2

import os
import glob
import sys
sys.path.insert(0, "../")

from pathlib import Path
import json

import pandas as pd
import seaborn as sns
import numpy as np

from matplotlib.ticker import LogLocator, LogFormatterMathtext, PercentFormatter

import matplotlib
from matplotlib import pyplot as plt
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

plt.rcParams.update({
    "font.size": 14,          # default font size
    "axes.titlesize": 16,     # title
    "axes.labelsize": 15,     # x/y labels
    "xtick.labelsize": 13,    # x tick labels
    "ytick.labelsize": 13,    # y tick labels
    "legend.fontsize": 13,
    "legend.title_fontsize": 14,
})

## Switch between motif length and series-length experiments

In [ ]:
max_memory = "8 GB"
# experiment = "motif length"

experiment = "series-length"

## Load data

In [ ]:
# Noise relative to the amplitude of the sine wave
amplitude = 5.0
x_col = "n" if experiment == "series-length" else "motif length"

def load_synthetic_results(result_dir, experiment):
  result_dir = Path(result_dir)
  stem = experiment.replace(" ", "-")

  summary_path = result_dir / f"{stem}_summary.csv"
  # jsonl_path = result_dir / f"{stem}_runs.jsonl"

  summary = pd.read_csv(summary_path)

  #with jsonl_path.open() as f:
  #    runs = [json.loads(line) for line in f]

  return summary # , runs


def aggregate_synthetic(summary, experiment, amplitude, backend_label):  
  agg = (
      summary
      .groupby([x_col, "noise"], as_index=False)
      .agg(
          time_mean=("time in s", "mean"),
          time_std=("time in s", "std"),
          f_score_mean=("f_score", "mean"),
          precision_mean=("precision", "mean"),
          recall_mean=("recall", "mean"),
          extent_mean=("selected extent", "mean"),
          ground_truth_extent=("ground truth extent", "mean"),
      )
  )

  agg["backend"] = backend_label
  agg["relative_noise"] = agg["noise"] / amplitude
  agg["relative_noise_pct"] = 100 * agg["relative_noise"]
  agg["relative_noise_label"] = agg["relative_noise_pct"].map(
      lambda value: f"{value:g}%"
  )

  threshold = 1e-2
  agg["relative_extent"] = np.where(
      agg["extent_mean"] < threshold,
      1.0,
      np.minimum(agg["extent_mean"] / agg["ground_truth_extent"], 1.0),
  )

  return agg


In [ ]:
scampi_summary = load_synthetic_results(
  "../tests/results/synthetic_scampi",
  experiment,
)


stumpy_summary = load_synthetic_results(
  "../tests/results/synthetic_stumpy",
  "series-length",
)

agg = aggregate_synthetic(
  scampi_summary,
  experiment,
  amplitude,
  backend_label="SCAMPI",
)

stumpy_agg = aggregate_synthetic(
  stumpy_summary,
  "series-length",
  amplitude,
  backend_label="STUMPY",
)

agg_all = pd.concat([agg, stumpy_agg], ignore_index=True)
agg_all.head()

In [ ]:
selected_noise = ["0%", "5%", "10%", "15%", "20%", "40%"]

# ---------------------------------------------------------------------
# Data
# ---------------------------------------------------------------------

scampi_df = agg_all[
    (agg_all["backend"] == "SCAMPI")
    & (agg_all["relative_noise_label"].isin(selected_noise))
].sort_values(x_col)

stumpy_df = agg_all[
    agg_all["backend"] == "STUMPY"
].sort_values(x_col)

# ---------------------------------------------------------------------
# Labels
# ---------------------------------------------------------------------

xlabel = {
    "series-length": "Series length",
    "motif-length": "Motif length",
}.get(experiment, x_col)


palette = sns.color_palette(
    "colorblind",
    n_colors=len(selected_noise),
)

# ---------------------------------------------------------------------
# Plot
# ---------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(5, 4))

sns.lineplot(
    data=scampi_df,
    x=x_col,
    y="time_mean",
    hue="relative_noise_label",
    hue_order=selected_noise,
    palette=palette,
    marker="o",
    markersize=6,
    linewidth=2.6,
    ax=ax,
)

if experiment == "series-length":
    sns.lineplot(
        data=stumpy_df,
        x=x_col,
        y="time_mean",
        color="black",
        linestyle=":",
        marker="s",
        markersize=6,
        linewidth=2.0,
        label=r"Exact$^*$" + "\n"+ r"($\mathcal{O}(n^2)$)",
        ax=ax,
    )

# ---------------------------------------------------------------------
# Axes
# ---------------------------------------------------------------------

ax.set_yscale("log")
ax.set_xscale("log")

ax.yaxis.set_major_locator(LogLocator(base=10))
ax.yaxis.set_major_formatter(LogFormatterMathtext())

ax.grid(
    axis="y",
    which="major",
    alpha=0.3,
    linewidth=0.6,
)

sns.despine()

ax.set_xlabel(xlabel+ " (log-scale)", fontsize=12)
ax.set_ylabel("Runtime (s, log-scale)", fontsize=12)

ax.set_title(
    rf"Runtime ($\delta=0.1$, {max_memory})",    # vs. {xlabel.lower()}
    fontsize=15,
    pad=10,
)

ax.tick_params(
    axis="both",
    labelsize=11,
)

# ---------------------------------------------------------------------
# Legend
# ---------------------------------------------------------------------

ax.legend(
    title="Relative noise",
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=False,
    fontsize=10,
    title_fontsize=10,
    handlelength=2.0,
    borderaxespad=0.0,
)

# ---------------------------------------------------------------------
# Layout
# ---------------------------------------------------------------------

fig.tight_layout()

fig.savefig(
    f"images/synthetic_runtime_lineplot_{experiment.replace(' ', '-')}.pdf",
    bbox_inches="tight",
)

plt.show()

In [ ]:
ted_noise = ["0%", "5%", "10%", "15%", "20%", "40%"]

plot_df = agg[
    agg["relative_noise_label"].isin(selected_noise)
]

# ---------------------------------------------------------------------
# Labels
# ---------------------------------------------------------------------

xlabel = {
    "series-length": "Series length",
    "motif-length": "Motif length",
}.get(experiment, x_col)


# ---------------------------------------------------------------------
# Plot
# ---------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(5, 4))

sns.lineplot(
    data=plot_df,
    x=x_col,
    y="relative_extent",
    hue="relative_noise_label",
    hue_order=selected_noise,
    palette=palette,
    marker="s",
    markersize=6,
    linewidth=2.6,
    ax=ax,
)

# ---------------------------------------------------------------------
# Axes
# ---------------------------------------------------------------------

ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))

ax.grid(
    axis="y",
    which="major",
    alpha=0.3,
    linewidth=0.6,
)

sns.despine()

ax.set_xlabel(xlabel, fontsize=12)
ax.set_ylabel("Relative extent\n(in percent of best)", fontsize=12)

ax.set_title(
    rf"Extent vs. {xlabel.lower()} ($\delta=0.1$, {max_memory})",
    fontsize=15,
    pad=10,
)

ax.tick_params(
    axis="both",
    labelsize=11,
)

# ---------------------------------------------------------------------
# Legend
# ---------------------------------------------------------------------

ax.legend(
    title="Relative noise",
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=False,
    fontsize=10,
    title_fontsize=10,
    handlelength=2.0,
    borderaxespad=0.0,
)

# ---------------------------------------------------------------------
# Layout
# ---------------------------------------------------------------------

fig.tight_layout()

fig.savefig(
    f"images/synthetic_quality2_lineplot_{experiment.replace(' ', '-')}.pdf",
    bbox_inches="tight",
)

plt.show()

In [ ]:
selected_noise = ["0%", "5%", "10%", "15%", "20%", "40%"]
plot_df = agg[agg["relative_noise_label"].isin(selected_noise)]

fig, ax = plt.subplots(figsize=(5, 4))

sns.barplot(
    data=plot_df,
    x="relative_noise_label",
    y="f_score_mean",
    #hue="relative_noise_label",
    ax=ax,
)

ax.set_xlabel("Relative Noise Level")
ax.set_ylabel("F-score")
ax.set_title(rf"F-score vs Noise Levels ($\delta=0.1$, {max_memory})")

plt.xticks(rotation=0)
plt.tight_layout()
#plt.savefig(
#    f"images/synthetic_quality_noise_barplot_{experiment.replace(' ','-')}.pdf",
#    bbox_inches="tight",
#)
plt.show()

In [ ]:
selected_noise = ["0%", "5%", "10%", "15%", "20%", "40%"]
plot_df = agg[agg["relative_noise_label"].isin(selected_noise)]

fig, ax = plt.subplots(figsize=(5, 4))

sns.barplot(
    data=plot_df,
    x="relative_noise_label",
    y="relative_extent",
    #hue="relative_noise_label",
    ax=ax,
)

ax.set_xlabel("Relative Extent Level")
ax.set_ylabel("Relative Extent")
ax.set_title(f"Relative Extent vs Noise Levels (Delta=0.1, {max_memory})")

plt.xticks(rotation=0)
plt.tight_layout()
#plt.savefig(
#    f"images/synthetic_extent_noise_barplot_{experiment.replace(' ','-')}.pdf",
#    bbox_inches="tight",
#)
plt.show()